In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import streamlit as st
import scipy.stats as sps


In [31]:
amazon_df=pd.read_csv('Amazon Sale Report.csv')
pd.set_option('display.max_columns',None)
pd.set_option('display.expand_frame_repr',False)

C:\Users\metr\AppData\Local\Temp\ipykernel_3188\4100434043.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  amazon_df=pd.read_csv('Amazon Sale Report.csv')


In [32]:
amazon_df.columns=amazon_df.columns.str.replace('-',' ').str.lower().str.strip()

In [ ]:
amazon_numerical=['date','qty','amount','ship postal code']
amazon_categorical=['status','fulfilment','sales channel','ship service level','style',
                    'sku','category','courier status','currency','ship city','ship state',
                    'ship country','fulfilled by','b2b','promotion ids']
#updates=date to datetime,style and sku=details about products=object,ship postal code to int64
# ,promotion ids to object,b2b to boolin
#target = status
clean_df=amazon_df.drop(columns=['index','sku','courier status','currency','ship country','fulfilled by','unnamed: 22'])
clean_df['date']=pd.to_datetime(clean_df['date'],format='%Y/%m/%d %H/%M/%S',errors='coerce')
clean_df.head()
#i wanna to extract column feature like(sum of counts for order id)



,order id,date,status,fulfilment,sales channel,ship service level,style,category,size,asin,qty,amount,ship city,ship state,ship postal code,promotion ids,b2b
0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,Set,S,B09KXVBD7Z,0,647.62,MUMBAI,MAHARASHTRA,400081.0,NaN,False
1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,kurta,3XL,B09K3WFS32,1,406.00,BENGALURU,KARNATAKA,560085.0,Amazon PLCC Free-Financing Universal Merchant ...,False
2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,kurta,XL,B07WV4JV4D,1,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN Core Free Shipping 2015/04/08 23-48-5-108,True
3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,Western Dress,L,B099NRCT7B,0,753.33,PUDUCHERRY,PUDUCHERRY,605008.0,NaN,False
4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,Top,3XL,B098714BZP,1,574.00,CHENNAI,TAMIL NADU,600073.0,NaN,False


In [54]:
clean_df.describe()

,date,qty,amount,ship postal code
count,0,128975.000000,121180.000000,128942.000000
mean,NaT,0.904431,648.561465,463966.236509
min,NaT,0.000000,0.000000,110001.000000
25%,NaT,1.000000,449.000000,382421.000000
50%,NaT,1.000000,605.000000,500033.000000
75%,NaT,1.000000,788.000000,600024.000000
max,NaT,15.000000,5584.000000,989898.000000
std,NaN,0.313354,281.211687,191476.764941


In [34]:
clean_df.info()
#(date to datetime,)
clean_df['promotion ids'].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   order id            128975 non-null  object 
 1   date                128975 non-null  object 
 2   status              128975 non-null  object 
 3   fulfilment          128975 non-null  object 
 4   sales channel       128975 non-null  object 
 5   ship service level  128975 non-null  object 
 6   style               128975 non-null  object 
 7   category            128975 non-null  object 
 8   size                128975 non-null  object 
 9   asin                128975 non-null  object 
 10  qty                 128975 non-null  int64  
 11  amount              121180 non-null  float64
 12  ship city           128942 non-null  object 
 13  ship state          128942 non-null  object 
 14  ship postal code    128942 non-null  float64
 15  promotion ids       79822 non-null

array([nan,
       'Amazon PLCC Free-Financing Universal Merchant AAT-WNKTBO3K27EJC,Amazon PLCC Free-Financing Universal Merchant AAT-QX3UCCJESKPA2,Amazon PLCC Free-Financing Universal Merchant AAT-5QQ7BIYYQEDN2,Amazon PLCC Free-Financing Universal Merchant AAT-DSJ2QRXXWXVMQ,Amazon PLCC Free-Financing Universal Merchant AAT-CXJHMC2YJUK76,Amazon PLCC Free-Financing Universal Merchant AAT-CC4FAVTYR4X7C,Amazon PLCC Free-Financing Universal Merchant AAT-XXRCW6NZEPZI4,Amazon PLCC Free-Financing Universal Merchant AAT-CXNSLNBROFDW4,Amazon PLCC Free-Financing Universal Merchant AAT-R7GXNZWISTRFA,Amazon PLCC Free-Financing Universal Merchant AAT-WSJLDN3X7KEMO,Amazon PLCC Free-Financing Universal Merchant AAT-VL6FGQVGQVXUS,Amazon PLCC Free-Financing Universal Merchant AAT-EOKPWFWYW7Y6I,Amazon PLCC Free-Financing Universal Merchant AAT-ZYL5UPUNW6T62,Amazon PLCC Free-Financing Universal Merchant AAT-XVPICCHRWDCAI,Amazon PLCC Free-Financing Universal Merchant AAT-ETXQ3XXWMRXBG,Amazon PLCC Free-Fin

In [72]:
clean_df.groupby(['status','ship service level'])['amount'].sum().reset_index().sort_values(by='status',ascending=True)


,status,ship service level,amount
0,Cancelled,Expedited,3726750.0
1,Cancelled,Standard,3192534.3
2,Pending,Expedited,268784.0
3,Pending,Standard,161487.0
4,Pending - Waiting for Pick Up,Standard,192138.0
5,Shipped,Expedited,50289649.0
6,Shipped,Standard,34606.0
7,Shipped - Damaged,Standard,1136.0
8,Shipped - Delivered to Buyer,Standard,18650815.0
9,Shipped - Lost in Transit,Standard,1997.0


# relations between target and category and promotion ids agg=sum(amount),count(order id)

In [73]:
clean_df[clean_df['status']=='Cancelled'].groupby(['status','category','promotion ids'])['amount'].sum().reset_index().sort_values(by='amount',ascending=False)
#clean_df[clean_df['status']=='Cancelled'].groupby(['status','category','promotion ids'])['order id'].count().reset_index().sort_values(by='order id',ascending=False)
clean_df[clean_df['status']=='Cancelled'].groupby(['status','category','promotion ids']).agg({'order id':'count',
                                                                                              'amount':'sum'}).reset_index().sort_values(by='amount',ascending=False)

,status,category,promotion ids,order id,amount
4,Cancelled,Set,IN Core Free Shipping 2015/04/08 23-48-5-108,111,9474.0
7,Cancelled,kurta,IN Core Free Shipping 2015/04/08 23-48-5-108,121,6359.0
5,Cancelled,Top,IN Core Free Shipping 2015/04/08 23-48-5-108,25,1509.0
6,Cancelled,Western Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,33,735.0
3,Cancelled,Set,Duplicated A12RHGVGRWOT3S 1560498941486,1,0.0
2,Cancelled,Ethnic Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,2,0.0
1,Cancelled,Bottom,IN Core Free Shipping 2015/04/08 23-48-5-108,1,0.0
0,Cancelled,Blouse,IN Core Free Shipping 2015/04/08 23-48-5-108,1,0.0


In [74]:
clean_df[clean_df['status']=='Pending'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                            'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
52,Pending,Set,IN Core Free Shipping 2015/04/08 23-48-5-108,98539.0,115
150,Pending,kurta,IN Core Free Shipping 2015/04/08 23-48-5-108,49575.0,103
98,Pending,Western Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,24474.0,32
67,Pending,Top,IN Core Free Shipping 2015/04/08 23-48-5-108,14877.0,25
22,Pending,Set,Amazon PLCC Free-Financing Universal Merchant ...,6750.0,7
16,Pending,Set,Amazon PLCC Free-Financing Universal Merchant ...,4442.0,5
3,Pending,Ethnic Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,3477.0,4
36,Pending,Set,Amazon PLCC Free-Financing Universal Merchant ...,3466.0,4
44,Pending,Set,Amazon PLCC Free-Financing Universal Merchant ...,3304.0,3
30,Pending,Set,Amazon PLCC Free-Financing Universal Merchant ...,3057.0,4


In [75]:
clean_df[clean_df['status']=='Pending - Waiting for Pick Up'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                                  'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
11,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,6380.0,6
22,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,5570.0,6
82,Pending - Waiting for Pick Up,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,5125.0,7
14,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,4189.0,3
25,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,3560.0,4
81,Pending - Waiting for Pick Up,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,3519.0,5
48,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,3052.0,3
58,Pending - Waiting for Pick Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,3039.0,3
122,Pending - Waiting for Pick Up,kurta,Amazon PLCC Free-Financing Universal Merchant ...,3005.0,5
84,Pending - Waiting for Pick Up,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,3004.0,4


In [76]:
clean_df[clean_df['status']=='Shipped'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                            'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
12,Shipped,Set,IN Core Free Shipping 2015/04/08 23-48-5-108,17727576.0,20784
40,Shipped,kurta,IN Core Free Shipping 2015/04/08 23-48-5-108,7329934.0,15161
30,Shipped,Western Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,3943106.0,5044
21,Shipped,Top,IN Core Free Shipping 2015/04/08 23-48-5-108,2004522.0,3615
8,Shipped,Ethnic Dress,IN Core Free Shipping 2015/04/08 23-48-5-108,300501.0,385
38,Shipped,kurta,Duplicated A12RHGVGRWOT3S 1560498941486,183131.0,465
2,Shipped,Blouse,IN Core Free Shipping 2015/04/08 23-48-5-108,171832.0,308
39,Shipped,kurta,Duplicated AYTJSBA8ZOP16 1567159860988,158285.0,399
9,Shipped,Saree,IN Core Free Shipping 2015/04/08 23-48-5-108,73607.0,92
34,Shipped,Western Dress,VPC-44571-38707197 Coupon,56210.0,66


In [77]:
clean_df[clean_df['status']=='Shipped - Damaged'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                      'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
0,Shipped - Damaged,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,1136.0,1


In [78]:
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                                 'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
2815,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,164634.0,208
2811,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,124661.0,147
1535,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,114244.0,140
1532,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,80564.0,102
1126,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,77906.0,82
1225,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,76029.0,92
1189,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,72528.0,85
2828,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,63716.0,76
8782,Shipped - Delivered to Buyer,kurta,Amazon PLCC Free-Financing Universal Merchant ...,63669.0,141
2837,Shipped - Delivered to Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,62327.0,81


In [79]:
clean_df[clean_df['status']=='Shipped - Lost in Transit'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                              'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
1,Shipped - Lost in Transit,Set,Amazon PLCC Free-Financing Universal Merchant ...,999.0,1
3,Shipped - Lost in Transit,kurta,Amazon PLCC Free-Financing Universal Merchant ...,998.0,2
0,Shipped - Lost in Transit,Set,Amazon PLCC Free-Financing Universal Merchant ...,0.0,1
2,Shipped - Lost in Transit,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,0.0,1


In [80]:
clean_df[clean_df['status']=='Shipped - Out for Delivery'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                               'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
8,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,2095.0,2
10,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1399.0,1
1,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1398.0,1
6,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1388.0,1
0,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1163.0,1
5,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1163.0,1
12,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1115.0,1
7,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,1083.0,2
27,Shipped - Out for Delivery,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,899.0,1
11,Shipped - Out for Delivery,Set,Amazon PLCC Free-Financing Universal Merchant ...,852.0,1


In [81]:
clean_df[clean_df['status']=='Shipped - Picked Up'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                       'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
86,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,11981.0,11
84,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,9035.0,10
91,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,7425.0,7
53,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,6465.0,7
87,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,5736.0,6
93,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,5521.0,7
112,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,5282.0,5
59,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,5281.0,6
295,Shipped - Picked Up,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,5080.0,7
137,Shipped - Picked Up,Set,Amazon PLCC Free-Financing Universal Merchant ...,4924.0,6


In [82]:
clean_df[clean_df['status']=='Shipped - Rejected by Buyer'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                                'order id':'count'}).reset_index().sort_values(by='amount',ascending=False).head(35)


,status,category,promotion ids,amount,order id
1,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,967.0,1
8,Shipped - Rejected by Buyer,Western Dress,Amazon PLCC Free-Financing Universal Merchant ...,899.0,1
0,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,790.0,1
5,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,788.0,1
3,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,725.0,1
2,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,654.0,1
4,Shipped - Rejected by Buyer,Set,Amazon PLCC Free-Financing Universal Merchant ...,568.0,1
9,Shipped - Rejected by Buyer,kurta,Amazon PLCC Free-Financing Universal Merchant ...,568.0,1
10,Shipped - Rejected by Buyer,kurta,Amazon PLCC Free-Financing Universal Merchant ...,521.0,1
6,Shipped - Rejected by Buyer,Top,Amazon PLCC Free-Financing Universal Merchant ...,518.0,1


In [83]:
clean_df[clean_df['status']=='Shipped - Returning to Seller'].groupby(['status','category','promotion ids']).agg({'amount':'sum',
                                                                                                                  'order id':'count'}).reset_index().sort_values(by='amount',ascending=False)

,status,category,promotion ids,amount,order id
53,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,4285.0,4
40,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,4003.0,4
21,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,3866.0,4
49,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,2726.0,3
46,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,2586.0,3
...,...,...,...,...,...
111,Shipped - Returning to Seller,kurta,Amazon PLCC Free-Financing Universal Merchant ...,333.0,1
93,Shipped - Returning to Seller,kurta,Amazon PLCC Free-Financing Universal Merchant ...,325.0,1
92,Shipped - Returning to Seller,kurta,Amazon PLCC Free-Financing Universal Merchant ...,292.0,1
1,Shipped - Returning to Seller,Set,Amazon PLCC Free-Financing Universal Merchant ...,0.0,1


# relations between target and(ship city,ship state).agg(qty)

In [84]:
clean_df['status'].unique()

array(['Cancelled', 'Shipped - Delivered to Buyer', 'Shipped',
       'Shipped - Returned to Seller', 'Shipped - Rejected by Buyer',
       'Shipped - Lost in Transit', 'Shipped - Out for Delivery',
       'Shipped - Returning to Seller', 'Shipped - Picked Up', 'Pending',
       'Pending - Waiting for Pick Up', 'Shipped - Damaged', 'Shipping'],
      dtype=object)

In [85]:
clean_df[clean_df['status']=='Cancelled'].groupby(
    ['status','ship city','ship state']
    ).agg({'qty':'sum',
          'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)


,status,ship city,ship state,qty,amount
286,Cancelled,BENGALURU,KARNATAKA,487,496664.99
1029,Cancelled,HYDERABAD,TELANGANA,387,449242.82
1693,Cancelled,MUMBAI,MAHARASHTRA,270,306899.80
516,Cancelled,CHENNAI,TAMIL NADU,244,277771.74
1883,Cancelled,NEW DELHI,DELHI,223,281636.83
...,...,...,...,...,...
3022,Cancelled,vadodara,Gujarat,0,0.00
3021,Cancelled,ukhra,WEST BENGAL,0,0.00
3019,Cancelled,trichy,TAMIL NADU,0,546.67
3018,Cancelled,trichirapalli,TAMIL NADU,0,266.67


In [86]:
clean_df[clean_df['status']=='Shipped - Delivered to Buyer'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                               'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
352,Shipped - Delivered to Buyer,BENGALURU,KARNATAKA,2336,1463928.0
1296,Shipped - Delivered to Buyer,HYDERABAD,TELANGANA,1685,1078250.0
2180,Shipped - Delivered to Buyer,MUMBAI,MAHARASHTRA,1420,881790.0
2452,Shipped - Delivered to Buyer,NEW DELHI,DELHI,1317,865849.0
654,Shipped - Delivered to Buyer,CHENNAI,TAMIL NADU,1075,657872.0
...,...,...,...,...,...
3997,Shipped - Delivered to Buyer,"sudarsan nagar, madambakkam",TAMIL NADU,1,751.0
3996,Shipped - Delivered to Buyer,"somanur, coimbatore",TAMIL NADU,1,523.0
3995,Shipped - Delivered to Buyer,singrauli,MADHYA PRADESH,1,665.0
3994,Shipped - Delivered to Buyer,sillod,MAHARASHTRA,1,376.0


In [87]:
clean_df[clean_df['status']=='Shipped'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                          'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
614,Shipped,BENGALURU,KARNATAKA,7339,4723666.0
2304,Shipped,HYDERABAD,TELANGANA,5020,3250847.0
3754,Shipped,MUMBAI,MAHARASHTRA,3839,2437985.0
4189,Shipped,NEW DELHI,DELHI,3532,2345059.0
1155,Shipped,CHENNAI,TAMIL NADU,3469,2074972.0
...,...,...,...,...,...
6931,Shipped,unai,Gujarat,1,329.0
17,Shipped,ACHHNERA,UTTAR PRADESH,1,1163.0
16,Shipped,ACHARAPAKKAM,TAMIL NADU,1,523.0
15,Shipped,ACHANTA,ANDHRA PRADESH,1,542.0


In [88]:
clean_df[clean_df['status']=='Shipped - Returned to Seller'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                               'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
75,Shipped - Returned to Seller,BENGALURU,KARNATAKA,115,70001.0
255,Shipped - Returned to Seller,HYDERABAD,TELANGANA,97,65501.0
479,Shipped - Returned to Seller,NEW DELHI,DELHI,84,55364.0
130,Shipped - Returned to Seller,CHENNAI,TAMIL NADU,57,33167.0
539,Shipped - Returned to Seller,PUNE,MAHARASHTRA,49,33253.0
...,...,...,...,...,...
4,Shipped - Returned to Seller,AHMADNAGAR,MAHARASHTRA,1,775.0
10,Shipped - Returned to Seller,"ALAGAPPAPURAM,kanyakumari district",TAMIL NADU,1,560.0
723,Shipped - Returned to Seller,VISHRAMPUR,CHHATTISGARH,1,799.0
724,Shipped - Returned to Seller,Vaishali,Bihar,1,654.0


In [89]:
clean_df[clean_df['status']=='Shipped - Rejected by Buyer'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                              'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
0,Shipped - Rejected by Buyer,AJMER,RAJASTHAN,1,790.0
1,Shipped - Rejected by Buyer,CHENNAI,TAMIL NADU,1,725.0
2,Shipped - Rejected by Buyer,FARIDABAD,HARYANA,1,788.0
3,Shipped - Rejected by Buyer,LUCKNOW,UTTAR PRADESH,1,899.0
4,Shipped - Rejected by Buyer,MUMBAI,MAHARASHTRA,1,518.0
5,Shipped - Rejected by Buyer,NASHIK,MAHARASHTRA,1,654.0
6,Shipped - Rejected by Buyer,PUNE,MAHARASHTRA,1,521.0
7,Shipped - Rejected by Buyer,RAMANAGARA,KARNATAKA,1,568.0
8,Shipped - Rejected by Buyer,URAN ISLAMPUR,MAHARASHTRA,1,297.0
9,Shipped - Rejected by Buyer,VISAKHAPATNAM,ANDHRA PRADESH,1,568.0


In [90]:
clean_df[clean_df['status']=='Shipped - Lost in Transit'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                            'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
0,Shipped - Lost in Transit,BENGALURU,KARNATAKA,2,0.0
2,Shipped - Lost in Transit,Bengaluru,KARNATAKA,2,998.0
1,Shipped - Lost in Transit,BHINMAL,RAJASTHAN,1,999.0


In [91]:
clean_df[clean_df['status']=='Shipped - Out for Delivery'].groupby(
                                            ['status','ship city','ship state']).agg({'qty':'sum',
                                            'amount':'sum'},
                                            ).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
3,Shipped - Out for Delivery,CHENNAI,TAMIL NADU,4,2158.0
7,Shipped - Out for Delivery,HYDERABAD,TELANGANA,4,3340.0
1,Shipped - Out for Delivery,BENGALURU,KARNATAKA,3,2232.0
20,Shipped - Out for Delivery,PUNE,MAHARASHTRA,2,654.0
14,Shipped - Out for Delivery,MUMBAI,MAHARASHTRA,2,2095.0
4,Shipped - Out for Delivery,DIMAPUR,NAGALAND,1,371.0
5,Shipped - Out for Delivery,ELURU,ANDHRA PRADESH,1,1388.0
0,Shipped - Out for Delivery,BANGALORE,KARNATAKA,1,735.0
2,Shipped - Out for Delivery,BRAJARAJNAGAR,ODISHA,1,831.0
8,Shipped - Out for Delivery,JAMSHEDPUR,JHARKHAND,1,792.0


In [92]:
clean_df[clean_df['status']=='Shipped - Returning to Seller'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                                'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
7,Shipped - Returning to Seller,BENGALURU,KARNATAKA,10,5499.0
13,Shipped - Returning to Seller,CHENNAI,TAMIL NADU,6,3080.0
21,Shipped - Returning to Seller,DIMAPUR,NAGALAND,6,5224.0
60,Shipped - Returning to Seller,NEW DELHI,DELHI,6,3101.0
52,Shipped - Returning to Seller,MUMBAI,MAHARASHTRA,5,4795.0
...,...,...,...,...,...
79,Shipped - Returning to Seller,SURAT,Gujarat,1,574.0
80,Shipped - Returning to Seller,THANE,MAHARASHTRA,1,648.0
82,Shipped - Returning to Seller,VARANASI,UTTAR PRADESH,1,999.0
83,Shipped - Returning to Seller,VIJAYAWADA,ANDHRA PRADESH,1,335.0


In [93]:
clean_df[clean_df['status']=='Shipped - Picked Up'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                      'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
143,Shipped - Picked Up,HYDERABAD,TELANGANA,59,40194.0
38,Shipped - Picked Up,BENGALURU,KARNATAKA,55,34489.0
64,Shipped - Picked Up,CHENNAI,TAMIL NADU,43,26457.0
268,Shipped - Picked Up,NEW DELHI,DELHI,41,29288.0
244,Shipped - Picked Up,MUMBAI,MAHARASHTRA,27,20799.0
...,...,...,...,...,...
425,Shipped - Picked Up,panchkula,HARYANA,1,599.0
426,Shipped - Picked Up,"uluberia, howrah",WEST BENGAL,1,301.0
6,Shipped - Picked Up,ALAPPUZHA,KERALA,1,387.0
3,Shipped - Picked Up,AHMADNAGAR,MAHARASHTRA,1,292.0


In [94]:
clean_df[clean_df['status']=='Pending'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                          'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
23,Pending,BENGALURU,KARNATAKA,50,36550.0
77,Pending,HYDERABAD,TELANGANA,44,33445.0
144,Pending,MUMBAI,MAHARASHTRA,33,18612.0
163,Pending,NEW DELHI,DELHI,28,19194.0
39,Pending,CHENNAI,TAMIL NADU,26,13662.0
...,...,...,...,...,...
13,Pending,Agra,UTTAR PRADESH,1,908.0
12,Pending,AZAMGARH,UTTAR PRADESH,1,735.0
264,Pending,yazali,ARUNACHAL PRADESH,1,487.0
10,Pending,ASANSOL,WEST BENGAL,1,999.0


In [95]:
clean_df[clean_df['status']=='Pending - Waiting for Pick Up'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                                'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
12,Pending - Waiting for Pick Up,BENGALURU,KARNATAKA,25,16635.0
58,Pending - Waiting for Pick Up,HYDERABAD,TELANGANA,24,16641.0
98,Pending - Waiting for Pick Up,NEW DELHI,DELHI,12,8645.0
31,Pending - Waiting for Pick Up,CHENNAI,TAMIL NADU,11,7745.0
86,Pending - Waiting for Pick Up,MUMBAI,MAHARASHTRA,11,6319.0
...,...,...,...,...,...
141,Pending - Waiting for Pick Up,ZIRAKPUR,PUNJAB,1,461.0
143,Pending - Waiting for Pick Up,delhi,DELHI,1,613.0
144,Pending - Waiting for Pick Up,derabassi,Punjab,1,845.0
145,Pending - Waiting for Pick Up,kolhapur,MAHARASHTRA,1,1127.0


In [96]:
clean_df[clean_df['status']=='Shipped - Damaged'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                    'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
0,Shipped - Damaged,CHENNAI,TAMIL NADU,1,1136.0


In [97]:
clean_df[clean_df['status']=='Shipping'].groupby(['status','ship city','ship state']).agg({'qty':'sum',
                                                                                                    'amount':'sum'}).reset_index().sort_values(by='qty',ascending=False)

,status,ship city,ship state,qty,amount
0,Shipping,Surat,Gujarat,8,0.0


In [98]:
def clean(x):
    x=x.str.replace('...','').str.strip()
    x=x.astype('int64')
    return x


# -------------------
# taxi mid project

In [99]:
taxi_df.info()
#vendor id to int64,lpep pickup datetime and lpep dropoff datetime to datetime,
#ratecodeid to int64,passenger count to int64,payment type to int64,trip type to int64
#

NameError: name 'taxi_df' is not defined

In [ ]:
taxi_df=pd.read_csv('taxi_tripdata.csv')
taxi_df.columns=taxi_df.columns.str.replace('_',' ').str.lower().str.strip()
taxi_numeric=['lpep pickup datetime','lpep dropoff datetime','ratecodeid',
              'pulocationID','dolocationid','passenger count','trip distance',
              'fare amount','extra','mta tax','tip amount','tolls amount',
              'improvement surcharge','total amount','payment type','trip type']
taxi_categoric=['store and fwd flag']
taxi_df.head()
#taxi_df[taxi_df['store and fwd flag']=='Y'].groupby('ratecodeid')['total amount'].sum().reset_index().sort_values(by='total amount',ascending=False)

C:\Users\metr\AppData\Local\Temp\ipykernel_176\1121496796.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  taxi_df=pd.read_csv('taxi_tripdata.csv')


,ratecodeid,total amount
0,1.0,2598.77
2,4.0,77.30
3,5.0,69.65
1,2.0,59.35


In [ ]:
def outliers_fun(x):
    x=x.dropna()
    q1=x.quantile(0.25)
    q3=x.quantile(0.75)
    iqr=q3-q1
    upper_limit=q3+1.5*iqr
    lower_limit=q1-1.5*iqr
    outliers=x[(x<lower_limit)|(x>upper_limit)]
    return round(len(outliers)/len(x)*100,2)
fill=['vendorid','ratecodeid']
report={
    i:outliers_fun(taxi_df[i])
    for i in fill
}
report
taxi_df['vendorid']=taxi_df['vendorid'].fillna(taxi_df['vendorid'].median())
taxi_df['ratecodeid']=taxi_df['ratecodeid'].fillna(taxi_df['ratecodeid'].mean())
################################################################################
filll=['passenger count','payment type','trip type']
reportt={
    i:outliers_fun(taxi_df[i])
    for i in filll
}
reportt
taxi_df['passenger count']=taxi_df['passenger count'].fillna(taxi_df['passenger count'].median())
taxi_df['payment type']=taxi_df['payment type'].fillna(taxi_df['payment type'].mean())
taxi_df['trip type']=taxi_df['trip type'].fillna(taxi_df['trip type'].mean())

taxi_df['passenger count']=taxi_df['passenger count'].astype('int64')
taxi_df['payment type']=taxi_df['payment type'].astype('int64')

In [ ]:
taxi_df['vendorid']=taxi_df['vendorid'].astype('int64')
taxi_df['ratecodeid']=taxi_df['ratecodeid'].astype('int64')

In [ ]:
taxi_df['ratecodeid'].dtype

dtype('int64')

In [ ]:

taxi_df['lpep dropoff datetime']=pd.to_datetime(taxi_df['lpep dropoff datetime'])
taxi_df['lpep pickup datetime']=pd.to_datetime(taxi_df['lpep pickup datetime'])


array([ 1.,  2., nan])